# 06 — v2.5 · The network between the 23 complexes (run spec v3 §1a revised 2026-09-29; §3 stages 3–4, §4)

`cc.start` on the headline variant + the contraction overrides, `run_id` from config (`v25_run001`; a NEW id after a second-pass
merge). The nodes are the complexes; inter-complex links are routed between the complex UNIONS (a path may leave any patch of
the source and enter any patch of the target — the unit field is the pointwise minimum over the member patches' cached fields,
so the CWD stage is a cache HIT). Order: G0 (n complexes) + GW5 → resistance → CWD → cutoff (inherited, D31) → network (G3) →
GW4 → G15 → near-optimality (G10/G22) → the counterfactual (D17, G13; cache HIT on the patch fields) → the **D25 second pass**
(merge + re-run if any inter-complex link is near-contiguous) → branches (D26; G9/G21) → `classify_links` (D23–D25; G18/G19) →
protection status (W11) → priority → `finish` (Layer A written by write_run). ~1 h; live internet not needed.

In [1]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/wolverine_refugia_connectivity/, below the repo root where
# config.py and the corridor engine modules sit. Same bootstrap as analyses/northern_connectivity/.
import sys, pathlib, json
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import numpy as np
import pandas as pd
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
for _m in (config, cp, cg, cc):
    importlib.reload(_m)

KEY = "wolverine"
CFG = config.CORRIDORS[KEY]
HERE = ROOT / "analyses" / "wolverine_refugia_connectivity"

In [2]:
V25 = CFG["v25"]
OV = cc._merge(CFG["variants"][CFG["headline_variant"]], V25["overrides"])
RUN_ID = V25["run_id"]                                      # <- bump after a second-pass merge (e.g. "v25_run002")
_ = cc.resolve(KEY, overrides=OV)                           # G8
A = cc.start(KEY, overrides=OV, label="v2.5: the network between the refugia complexes, withheld-terrain surface", run_id=RUN_ID, require_review=False)
summ = json.loads((HERE / "audit" / "audit_objects" / "complexes_summary.json").read_text())
cc.gate_g0(A, expect_names=summ["n_complexes"], expect_merges=0)
cc.gate_gw5(A)
ALLOW_DIRTY = False                                          # run spec v3 §9: the run record on a clean tree (True only for a smoke)
print("git:", A.rec["git"])
assert ALLOW_DIRTY or not A.rec["git"].get("dirty", True), "working tree dirty at run creation -- delete this run dir, commit, re-run (run spec v3 §9)"

run v25_run001 (v2.5: the network between the refugia complexes, withheld-terrain surface) -> output_data/corridors_wolverine/v25_run001
Y2Y — wolverine climate refugia: routing grid 4285x11038 @ 300 m = 17,240,740 routable cells (1,551,667 km²) -- whole window
  core refugia 318,822 km² in 1,845 8-connected patches; marginal 169,988 km²
    >=   100 km²:  239 patches holding  94.2% of core area
    >=   250 km²:  130 patches holding  88.8% of core area   <- node floor
    >=   500 km²:   67 patches holding  81.7% of core area
    >=  1000 km²:   36 patches holding  75.0% of core area
  node_names.csv applied: 130 display names overridden
  protected land (status layer, W11): existing PAs 201,672 km², proposed 179,528 km² | 43.5% of node land is protected
nodes: 130 core refugia patches >= 250 km² (282,996 km² of node land, numbered north -> south) | largest 78,454 km², smallest 254 km²
GW5 OK: 130 patches -> 23 complexes (8 multi-patch), membership sha cdd305f952e7
contraction (D-W3):

## Resistance → CWD (cache HIT on the 130 patch fields; the complexes' fields derived) → the inherited cutoff

In [3]:
cc.resistance(A)
cc.cost_distances(A)
cc.set_cutoff(A, CFG["cwd_cutoff_abs"])

resistance = O'Brien et al. 2025 movement cost, generic terrain rules withheld (elevation > 2300 m, slope > 30°, glaciers; Pither et al. 2023 S1 Table)
  movement_cost_terrain_withheld.tif @ 300 m, 'near' resampled (ordinal classes preserved exactly)
cost-weighted distance from 130 seed parts (cache HIT: 860bb26ec2dec42b_c, compact)
  cache 9.0 GB on disk, one field resident at a time
cwd_cutoff_abs = 13.6 (= 4.09 km of extra travel on open ground, D31) written into v25_run001/run_config.json


<corridors v25_run001 | 23 nodes | 4285x11038>

## Network (G3) → GW4 → G15

In [4]:
cc.corridor_network(A)
cc.gate_gw4(A)
cc.gate_g15(A)

  backup for Complex · C11 Refugium (Mudzenchoot, E) <-> Complex · C12 Refugia complex (Jasper): add Complex · C06 Refugia complex (Northern Rocky Mountains) <-> Complex · C12 Refugia complex (Jasper)  (498 = 1.10x)
  IRREPLACEABLE Complex · C12 Refugia complex (Jasper) <-> Complex · C20 Refugia complex (Selway Bitterroot)  (cheapest alternative 4.4x > beta=2.5) -- 132 node pairs stranded if lost
  backup for Complex · C19 Refugium (Skull-Odell, SE) <-> Complex · C20 Refugia complex (Selway Bitterroot): add Complex · C18 Refugium (Lee Metcalf, NW) <-> Complex · C20 Refugia complex (Selway Bitterroot)  (341 = 2.43x)
  IRREPLACEABLE Complex · C18 Refugium (Lee Metcalf, NW) <-> Complex · C23 Refugia complex (Yellowstone)  (cheapest alternative 4.1x > beta=2.5) -- 60 node pairs stranded if lost
  IRREPLACEABLE Complex · C20 Refugia complex (Selway Bitterroot) <-> Complex · C21 Refugium (Eagle Cap)  (cheapest alternative 3.6x > beta=2.5) -- 60 node pairs stranded if lost
  backup for Comple

,label_i,label_j,edge_class,in_mst,cost,centrality_cf,centrality_sp,rank_cf,rank_sp,rank_delta
edge_id,,,,,,,,,,
E011_019,Complex · C12 Refugia complex (Jasper),Complex · C20 Refugia complex (Selway Bitterroot),inter,True,89.982758,66.000000,132.0,1,1,0
E005_011,Complex · C06 Refugia complex (Northern Rocky ...,Complex · C12 Refugia complex (Jasper),inter,False,498.481262,41.544693,108.0,2,2,0
E005_006,Complex · C06 Refugia complex (Northern Rocky ...,"Complex · C07 Refugia complex (Sustut, E)",inter,True,33.970562,38.873360,88.0,3,3,0
E006_008,"Complex · C07 Refugia complex (Sustut, E)",Complex · C09 Refugium (Omineca),inter,True,214.421356,31.513897,18.0,4,21,-17
E019_020,Complex · C20 Refugia complex (Selway Bitterroot),Complex · C21 Refugium (Eagle Cap),inter,True,109.698486,30.000000,60.0,5,5,0
E017_022,"Complex · C18 Refugium (Lee Metcalf, NW)",Complex · C23 Refugia complex (Yellowstone),inter,True,90.982758,30.000000,60.0,6,5,1
E010_011,"Complex · C11 Refugium (Mudzenchoot, E)",Complex · C12 Refugia complex (Jasper),inter,True,453.368591,29.245077,24.0,7,10,-3
E008_010,Complex · C09 Refugium (Omineca),"Complex · C11 Refugium (Mudzenchoot, E)",inter,True,239.752304,28.964091,22.0,8,11,-3
E018_019,"Complex · C19 Refugium (Skull-Odell, SE)",Complex · C20 Refugia complex (Selway Bitterroot),inter,True,140.480225,24.937085,18.0,9,21,-12


## Near-optimality (G10, G22) → the counterfactual width (D17, G13)

In [5]:
cc.near_optimality(A)
cc.counterfactual_squeeze(A)

G10 OK: max residual on baseline least-cost paths = 6.87e-05 cost units
  G22 OK: tier breaks at the registered fractions of the cutoff; detour twin consistent
near-optimality surface over 17,240,740 routable cells (tiers = fixed slack breaks, fractions of the cutoff 13.6 = 4.1 km of detour on open ground)
  robust_core  (slack <=       2.27 =  cutoff/6):    11,463 km²
  frequent     (slack <=       6.81 =  cutoff/2):     8,848 km²
  occasional   (slack <=      13.62 =    cutoff):    10,699 km²
  beyond band  (routable, slack > cutoff): 1,520,656 km²
  wrote near_optimality.tif, near_opt_owner.tif, near_optimality_class.tif (+ owner legend)
D17 counterfactual: 3,213,998 cells (18.6% of routable) with cost >= 10 set to 1; banding every edge at cutoff 13.623
  counterfactual CWD from 130 seed parts (cache HIT: 657fc5f4a45b5321_cf_c)
    band 1/29 E000_001: 54408 cells @ 27.2459, 1 s (elapsed 1 s)
    band 2/29 E000_002: 98807 cells @ 27.2459, 2 s (elapsed 3 s)
    band 3/29 E001_005: 511

<corridors v25_run001 | 23 nodes | 4285x11038 | 29 edges | corridor 29,619 km²>

## D25 second pass

If any inter-complex link is still near-contiguous, the pair is merged (`cc.merge_complexes`) and this notebook is re-run from the
top with a NEW `RUN_ID`. Expected zero or few.

In [6]:
m2 = cc.second_pass_merges(A)
if len(m2):
    print(m2.to_string(index=False))
    print("-> cc.merge_complexes(KEY, [(i, j), ...]) with the pairs above; commit the audit objects; set RUN_ID to a new id; re-run this notebook")
assert not len(m2), "second-pass merges pending -- see above"

D25 second pass: 1 inter-complex link(s) near-contiguous -> merge and re-run (cc.merge_complexes)
 edge_id  complex_i  complex_j  lcp_len_cells  open_ground_width_med  crosses_cost_1000
E011_019         12         20           56.0                   78.0              False
-> cc.merge_complexes(KEY, [(i, j), ...]) with the pairs above; commit the audit objects; set RUN_ID to a new id; re-run this notebook


AssertionError: second-pass merges pending -- see above

## Route branches (D26; G9/G21) → link classes (D23–D25; G18/G19) → protection status (W11) → priority → finish

In [ ]:
cc.route_branches(A)
tt = pd.read_csv(A.run_dir / "class_truth_table.csv"); print(tt.to_string(index=False))
cc.secured_status(A)
cc.priority_surface(A)
cc.finish(A)
cc.runs(KEY)

## Done — next `07_v25_product.ipynb` (coverage, accounting, tables, the headline) then `08_director_outputs.ipynb`.